# SRQS8F7JWA9MZ

In [ ]:
# Imports
from foodcast.imports import *
os.chdir(find_project_root())
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, _, _, _ = DATA_DIR_3_x

# Settings
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'SRQS8F7JWA9MZ'
df_uncleaned = load_single_restaurant(loc_id)

In [ ]:
# Remapping was manual, but stored in a yaml now to stay organized
remapping_path = Path('scripts') / 'labeling' / 'remapping' / 'loc1_remappings.yaml'
with open(remapping_path, "r", encoding="utf-8") as f:
    remapping = yaml.load(f, Loader=yaml.FullLoader)

# Extract the parts of the yaml to do the programmatic relabeling
df_relabeled = fully_relabel_and_consolidate(
    df                        = df_uncleaned,
    name_changes              = remapping.get("name_changes", {}),
    modification_name_changes = remapping.get("modification_name_changes", []),
    vegan_list                = remapping.get("vegan_list", []),
    vegetarian_list           = remapping.get("vegetarian_list", []),
    meat_list                 = remapping.get("meat_list", []),
    alcohol_list              = remapping.get("alcoholic_drinks", []),
    drinks_list               = remapping.get("non_alcoholic_drinks", []),
    merch                     = remapping.get("merch_list", []),
    rare                      = remapping.get("rare_list", []) + remapping.get("uncommon_list", []),
    unknown                   = remapping.get("unknown_list", []),
    remove_categories         = ["Merch", "Drink", "Rare"],
)

# # Are the labels uniquely specified?
# display(df_relabeled.groupby('item_name')['vegan'].unique()) 

df_relabeled.to_parquet(DATA_DIR_3_1 / (loc_id + '_sales_and_menu.parquet'))

# Consolidate dishes

df_consolidated = rename_items(
    df = df_relabeled,
    name_changes = {
        'Gold Standard - Bacon' : [],
        'Gold Standard - Kale' : ["Vegan Gold Standard - Kale", "Meat Gold Standard - Kale"],
        'Gold Standard - Bacon & Kale' :[],
        'Gold Standard - Impossible' : ['Meat Gold Standard - Impossible', 'Vegan Gold Standard - Impossible'],
        'Beyond Burger' : ['Meat Beyond Burger', 'Vegan Beyond Burger'],
        'Impossible Patty Melt' : ['Meat Impossible Patty Melt', 'Vegan Impossible Patty Melt'],
        'The Alternative' : ['Meat The Alternative', 'Vegetarian The Alternative', 'Impossible The Alternative'],
        'Bottled Pop': df_relabeled.value_counts('item_name').filter(regex='Coca|Crod|Ginger|Soda').index.tolist(),
        'Canned Drinks': df_relabeled.value_counts('item_name').filter(regex='Coco|Mate|Zam|Croi').index.tolist(),
    }
)

# # There should be multiple labels for each item
# display(df_consolidated.groupby('item_name')['vegan'].unique()) 

df_consolidated.to_parquet(DATA_DIR_3_2 / (loc_id + '_sales_and_menu.parquet'))

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_label="Gold Standard - Impossible", legend_min=5.31, legend_max=12.75, shift_adjustment=0.02)

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Impossible")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Impossible")')['item_quantity'].sum())

plot_time_series_subset(
    df_uncleaned, 
    exposure=promo_date,
    freq='W',
    truncate=True)
plt.show()
plot_time_series_subset(
    df_uncleaned.query('item_name.str.contains("Impossible")'),
    exposure=promo_date, 
    freq='W', 
    truncate=False)
plt.show()

In [ ]:
# Price analysis
print(df_consolidated
      .query('~vegetarian')
      ['unit_price']
      .mean())

print((df_consolidated
       .query('~vegetarian')
       ['item_name']
       .nunique()) / (df_consolidated
                      ['item_name']
                      .nunique()))
plt.plot(df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['unit_price']
         .mean(), 
         df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"').resample('W')['item_name'].nunique() / df_consolidated.query('dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique(), 
         'o', 
         alpha=0.5)
plt.show()